<a href="https://colab.research.google.com/github/IamAbhinav01/RENET---Movie-Recomendation-System/blob/main/RENET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Hello World")

Hello World


GOING TO INSTALL THE postgresql && redis APT for colab -- vm version

iam refering this one for postgresql: https://colab.research.google.com/github/tensorflow/io/blob/master/docs/tutorials/postgresql.ipynb#scrollTo=YUj0878jPyz7

and for redis :https://colab.research.google.com/drive/1jPgmnGdlVPLQq3c9YqAsxc_gu6YReKaS?usp=sharing#scrollTo=-MfrA0Ykepy0

In [2]:
!sudo apt-get -y -qq update
!sudo apt-get -y -qq install postgresql
!sudo apt-get update -qq
!sudo apt-get install -y redis-server
%pip install redis

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
redis-server is already the newest version (5:6.0.16-1ubuntu1.1).
0 upgraded, 0 newly installed, 0 to remove and 137 not upgraded.


Installing the nesscery requirements

In [3]:
%pip install pandas==2.2.2 numpy==1.26.4 scikit-learn==1.5.0 implicit==0.7.2 sentence-transformers==3.0.1 faiss-cpu==1.8.0 lightgbm==4.4.0 SQLAlchemy==2.0.31 psycopg2-binary==2.9.9 python-dotenv==1.0.1 tqdm==4.66.4 scipy==1.13.1

why these => {
  implicit -- Collaborative filtering/recommendation algorithms
  sentence-transformers -- Text Embeddings
  lightgbm -- Gradient Boosting / ranking the models
  SQLAlchemy -- database ORM
}

STARTING THE SERVICES

In [4]:
!sudo service postgresql start
!service redis-server start


 * Starting PostgreSQL 14 database server
   ...done.
Starting redis-server: redis-server.


STATUS CHECKING

In [5]:
import redis

client = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

print("Redis:", client.ping())

Redis: True


In [6]:
!service postgresql status

14/main (port 5432): online


In [7]:
!ps aux | grep redis

root       14357  0.1  0.0  67212  6628 ?        Ssl  16:55   0:03 redis-server *:6379
root       23392  0.0  0.0   7372  3604 ?        S    17:26   0:00 /bin/bash -c ps aux | grep redis
root       23394  0.0  0.0   6480  2452 ?        S    17:26   0:00 grep redis


creating a PostgreSQL user and a PostgreSQL database

In [8]:
!sudo -u postgres psql -c "CREATE USER renet WITH PASSWORD 'renet';"

ERROR:  role "renet" already exists


In [9]:
!sudo -u postgres psql -c "CREATE DATABASE renet OWNER renet;"

ERROR:  database "renet" already exists


Connecting env variables

In [10]:
from google.colab import userdata
password = userdata.get('POSTGRES_PASSWORD')

In [24]:
env_credentials = f"""DATABASE_URL=postgresql://renet:{password}
REDIS_URL=redis://localhost:6379/0
"""

with open(".env", "w") as f:
    f.write(env_credentials)

print("Created .env")

Created .env


Found a Dataset called as MOVIELENS
implicit → collaborative filtering
sentence-transformers → content/text embeddings
faiss-cpu → vector similarity search
LightGBM → ranking/recommendation

In [13]:
!mkdir -p data
!cd data && curl -L -o ml-latest-small.zip https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!cd data && unzip -q ml-latest-small.zip
!rm data/ml-latest-small.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  955k  100  955k    0     0   814k      0  0:00:01  0:00:01 --:--:--  813k


In [26]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [27]:
DATABASE_URL = os.getenv("DATABASE_URL")
DATA_DIR = os.environ.get("DATA_DIR", "./data/ml-latest-small")

In [28]:
import pandas as pd
print("\n--- Dataset files ---")

for file in os.listdir(DATA_DIR):
    if file.endswith(".csv"):
        file_path = os.path.join(DATA_DIR, file)

        df = pd.read_csv(file_path)

        print(f"\n===== {file} =====")
        print(df.head(5))


--- Dataset files ---

===== links.csv =====
   movieId  imdbId   tmdbId
0        1  114709    862.0
1        2  113497   8844.0
2        3  113228  15602.0
3        4  114885  31357.0
4        5  113041  11862.0

===== ratings.csv =====
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931

===== movies.csv =====
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3           

DATABASE SCHEMA

                ARCHITECTURE OF DATABASE
                
                
                 PostgreSQL
                     │
       ┌─────────────┼─────────────┐
       │             │             │
     users         items      interactions
       │             │             │
       │             │             │
       └─────────────┴─────────────┘
                     │
              Recommendation
                  System

One user → many interactions
One movie → many interactions

CASCADE means PostgreSQL will also remove objects that depend on those tables

THE kind of table iam paling is {
   | id | title     | genres                       | primary_genre |
| -: | --------- | ---------------------------- | ------------- |
|  1 | Toy Story | Adventure|Animation|Children | Animation     |
|  2 | Jumanji   | Adventure|Children|Fantasy   | Adventure     |

}

and the interaction table is {
| id | user_id | item_id | rating | event_type | created_at |
| -: | ------: | ------: | -----: | ---------- | ---------- |
|  1 |       1 |       1 |    5.0 | rating     | ...        |
|  2 |       1 |       3 |    4.0 | rating     | ...        |
|  3 |       2 |       1 |    3.5 | rating     | ...        |
}


In [30]:
SQL_SCHEMA = """
DROP TABLE IF EXISTS interactions CASCADE;
DROP TABLE IF EXISTS items CASCADE;
DROP TABLE IF EXISTS users CASCADE;

CREATE TABLE users (
    id INTEGER PRIMARY KEY
);

CREATE TABLE items (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    genres TEXT NOT NULL,
    primary_genre TEXT NOT NULL
);

CREATE TABLE interactions (
    id SERIAL PRIMARY KEY,
    user_id INTEGER NOT NULL REFERENCES users(id),
    item_id INTEGER NOT NULL REFERENCES items(id),
    rating REAL NOT NULL,
    event_type TEXT NOT NULL DEFAULT 'rating',
    created_at TIMESTAMP NOT NULL DEFAULT now()
);

CREATE INDEX idx_interactions_user ON interactions(user_id);
CREATE INDEX idx_interactions_item ON interactions(item_id);

"""

DATA INGESTION

**MovieLens CSV files → transforms them → creates PostgreSQL tables → inserts the data.**

In [33]:
movies_path = os.path.join(DATA_DIR, "movies.csv")
ratings_path = os.path.join(DATA_DIR, "ratings.csv")

In [34]:
if not os.path.exists(movies_path) or not os.path.exists(ratings_path):
  print(f"ERROR: expected {movies_path} and {ratings_path}.")
  print("Run: bash data/download_data.sh   (or download MovieLens manually)")
  sys.exit(1)

In [35]:
movies = pd.read_csv(movies_path)
ratings = pd.read_csv(ratings_path)

In [36]:
movies["primary_genre"] = movies["genres"].str.split("|").str[0].fillna("Unknown")

sqlAlchemy -> https://docs.sqlalchemy.org/en/20/core/engines.html#sqlalchemy.create_engine

In [44]:
from sqlalchemy import create_engine,text

In [45]:
engine = create_engine(DATABASE_URL)

              INTEND TODO THIS STRUCTURE
              MovieLens
                  │
        ┌─────────┴─────────┐
        │                   │
    movies.csv          ratings.csv
        │                   │
        ▼                   ▼
     Pandas              Pandas
        │                   │
        ▼                   ├──────────────┐
     items                  │              │
        │                   ▼              ▼
        │                unique users    ratings
        │                   │              │
        ▼                   ▼              ▼
    PostgreSQL           users        interactions

In [47]:
with engine.begin() as conn:
    print("Creating schema...")

    for stmt in SQL_SCHEMA.split(";"):
        if stmt.strip():
            conn.execute(text(stmt))

    print(f"Loading {len(movies)} items...")

    movies[["movieId", "title", "genres", "primary_genre"]].rename(
        columns={"movieId": "id"}
    ).to_sql(
        "items",
        conn,
        if_exists="append",
        index=False
    )

    print(f"Loading {ratings.userId.nunique()} users...")

    pd.DataFrame(
        {"id": ratings.userId.unique()}
    ).to_sql(
        "users",
        conn,
        if_exists="append",
        index=False
    )

    print(f"Loading {len(ratings)} interactions...")

    ratings_out = ratings.rename(
        columns={
            "userId": "user_id",
            "movieId": "item_id"
        }
    )[["user_id", "item_id", "rating"]]

    ratings_out["event_type"] = "rating"

    ratings_out.to_sql(
        "interactions",
        conn,
        if_exists="append",
        index=False
    )

print("Done. Data loaded into Postgres.")

Creating schema...
Loading 9742 items...
Loading 610 users...
Loading 100836 interactions...
Done. Data loaded into Postgres.


In [48]:
POSITIVE_RATING_THRESHOLD = 4.0
FACTORS = 64
REGULARIZATION = 0.05
ITERATIONS = 20


In [50]:
MODELS_DIR = os.environ.get("MODELS_DIR", "./models")

In [51]:
os.makedirs(MODELS_DIR, exist_ok=True)
engine = create_engine(DATABASE_URL)

In [52]:
interactions = pd.read_sql("SELECT user_id, item_id, rating FROM interactions", engine)

In [53]:
interactions

,user_id,item_id,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


implicit feedbcak

In [55]:
interactions = interactions[interactions.rating >= POSITIVE_RATING_THRESHOLD]

In [56]:
interactions

,user_id,item_id,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100830,610,166528,4.0
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0


In [57]:
print(f"Training ALS on {len(interactions)} positive interactions "
f"({interactions.user_id.nunique()} users, {interactions.item_id.nunique()} items)")

Training ALS on 48580 positive interactions (609 users, 6298 items)


MovieLens ID → Matrix index

In [60]:
user_ids = interactions.user_id.astype("category")
item_ids = interactions.item_id.astype("category")

user-item interaction matrix

In [73]:
from scipy.sparse import coo_matrix
import numpy as np
from implicit.als import AlternatingLeastSquares
import pickle

In [64]:
user_item = coo_matrix( ( np.ones(len(interactions), dtype=np.float32), ( user_ids.cat.codes, item_ids.cat.codes ) ) ).tocsr()

In [67]:
print(user_item.shape)

(609, 6298)


In [69]:
model = AlternatingLeastSquares( factors=FACTORS, regularization=REGULARIZATION, iterations=ITERATIONS )

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Train the model

In [70]:
model.fit(user_item)

  0%|          | 0/20 [00:00<?, ?it/s]

In [71]:
artifact = {
        "model": model,
        "user_id_to_idx": {uid: idx for idx, uid in enumerate(user_ids.cat.categories)},
        "idx_to_user_id": dict(enumerate(user_ids.cat.categories)),
        "item_id_to_idx": {iid: idx for idx, iid in enumerate(item_ids.cat.categories)},
        "idx_to_item_id": dict(enumerate(item_ids.cat.categories)),
        "user_item_matrix": user_item,
    }

In [75]:
out_path = os.path.join(MODELS_DIR, "als_model.pkl")
with open(out_path, "wb") as f:
    pickle.dump(artifact, f)

print(f"Saved {out_path}")

Saved ./models/als_model.pkl
